In [ ]:
# Steps 1-2: the inputs, then Monocle 2 in R (z per gene). Skipped if OUT has them; otherwise ~20 min and 180 GB.
import copy, logging, os
import xml.etree.ElementTree as ET
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import SVG
from matplotlib import font_manager
from matplotlib.textpath import TextPath

OUT = os.environ.setdefault("OUT", "/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_08_xmens_pt_heatmaps")
!bash 1_inputs.sbatch && bash 2_monocle.sbatch
for d in ("fig4h", "fig5f", "mi"):
    os.makedirs(f"{OUT}/figures/{d}", exist_ok=True)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)  # no Helvetica Neue on the cluster
if os.path.exists("/System/Library/Fonts/HelveticaNeue.ttc"):        # on a Mac
    font_manager.fontManager.addfont("/System/Library/Fonts/HelveticaNeue.ttc")

In [ ]:
# Fig. 4H: the Yang et al. (2023) panel, rows in published order; SVG written directly: the 30 x 59.887 mm box,
# the colorbar (5 pt labels) and the labelled panel, live text in Helvetica Neue, placed by its metrics
z = pd.read_csv(f"{OUT}/z/fig4h_z.tsv", sep="\t", index_col=0).drop(columns="fit")
genes, Z = list(z.index), np.clip(z.to_numpy(float), -3, 3)
timepoint = pd.read_csv(f"{OUT}/inputs/cells_mouse.tsv.gz", sep="\t", usecols=["timepoint"])["timepoint"]
n_induced = f"{int((timepoint != 'uninduced').sum()):,}"
F, NS, FF = f"{OUT}/figures/fig4h", "http://www.w3.org/2000/svg", "'Helvetica Neue', Helvetica, Arial, sans-serif"
BOX_W, BOX_H, ZLIM = 30.0, 59.887, 3.0
PT, EPS = 25.4 / 72, 0.01                       # mm per point; mm each cell overlaps the next (no seams in viewers)
CMAP, norm = plt.get_cmap("RdBu_r"), mpl.colors.Normalize(-ZLIM, ZLIM)
fmt = lambda v: f"{v:.5f}".rstrip("0").rstrip(".")
ext = lambda s, pt, italic=False: TextPath((0, 0), s, size=pt * PT, prop=font_manager.FontProperties(
    family="Helvetica Neue", style="italic" if italic else "normal")).get_extents()

# the box: one rect per cell
nr, nc = Z.shape
cw, ch = BOX_W / nc, BOX_H / nr
rgba = CMAP(norm(Z))
out = ['<?xml version="1.0" encoding="UTF-8"?>',
       f'<svg xmlns="{NS}" width="{fmt(BOX_W)}mm" height="{fmt(BOX_H)}mm" viewBox="0 0 {fmt(BOX_W)} {fmt(BOX_H)}">',
       f"<!-- heatmap {BOX_W:g} x {BOX_H:g} mm; {nr} genes x {nc} pseudotime bins; z per gene, colour limits "
       f"±{ZLIM:g} -->",
       '<g id="heatmap" shape-rendering="crispEdges" stroke="none">']
for r in range(nr):
    for c in range(nc):
        w, h = cw + (EPS if c < nc - 1 else 0.0), ch + (EPS if r < nr - 1 else 0.0)
        out.append(f'<rect x="{fmt(c * cw)}" y="{fmt(r * ch)}" width="{fmt(w)}" height="{fmt(h)}" '
                   f'fill="{mpl.colors.to_hex(rgba[r, c])}"/>')
open(f"{F}/heatmap_monocle_induced_z3_box_30x59p887mm.svg", "w").write("\n".join(out + ["</g>", "</svg>"]) + "\n")

# the colorbar: the same 256 colours, ticks at whole z, 5 pt labels
label, fs, bw, bh = "z per gene", 5 * PT, 1.6, BOX_H * 0.5
tick_len, tick_w, tick_pad, lab_pad = 0.6 * PT, 0.25 * PT, 0.8 * PT, 1.2 * PT
ticks = np.arange(np.ceil(-ZLIM), np.floor(ZLIM) + 1).astype(int)
tick_labels = [("−" if t < 0 else "") + str(abs(t)) for t in ticks]
cap = ext("0", 5).height                        # digit height, for vertical centring
x0, y0 = 0.05, cap / 2 + 0.3                    # room for the end labels above and below the bar
tx = x0 + bw + tick_len + tick_pad              # left edge of the tick labels
lx = tx + max(ext(s, 5).x1 for s in tick_labels) + lab_pad + ext(label, 5).y1   # baseline of the rotated label
CB_W, CB_H = lx + abs(ext(label, 5).y0) + 0.3, bh + 2 * y0
yv = lambda v: y0 + bh * (1 - norm(v))          # z -> y (top = +3)
out = ['<?xml version="1.0" encoding="UTF-8"?>',
       f'<svg xmlns="{NS}" width="{fmt(CB_W)}mm" height="{fmt(CB_H)}mm" viewBox="0 0 {fmt(CB_W)} {fmt(CB_H)}">',
       f"<!-- colorbar for z per gene, ±{ZLIM:g}; bar {bw:g} x {bh:g} mm; {CMAP.N} colour levels -->",
       '<g id="colorbar" shape-rendering="crispEdges" stroke="none">']
for i in range(CMAP.N):                         # level i spans values [i, i+1)/256 of the range
    top, h = y0 + bh * (1 - (i + 1) / CMAP.N), bh / CMAP.N + (EPS if i > 0 else 0.0)
    out.append(f'<rect x="{fmt(x0)}" y="{fmt(top)}" width="{fmt(bw)}" height="{fmt(h)}" '
               f'fill="{mpl.colors.to_hex(CMAP(i))}"/>')
out += ["</g>", f'<g id="ticks" stroke="#666666" stroke-width="{fmt(tick_w)}">']
out += [f'<line x1="{fmt(x0 + bw)}" y1="{fmt(yv(t))}" x2="{fmt(x0 + bw + tick_len)}" y2="{fmt(yv(t))}"/>'
        for t in ticks]
out += ["</g>", f'<g id="tick-labels" font-family="{FF}" font-size="{fmt(fs)}" fill="#000000">']
out += [f'<text x="{fmt(tx)}" y="{fmt(yv(t) + cap / 2)}">{s}</text>' for t, s in zip(ticks, tick_labels)]
out += ["</g>", f'<text id="label" font-family="{FF}" font-size="{fmt(fs)}" fill="#000000" text-anchor="middle" '
        f'transform="translate({fmt(lx)},{fmt(y0 + bh / 2)}) rotate(-90)">{label}</text>', "</svg>"]
open(f"{F}/heatmap_monocle_z3_colorbar_5pt_30x59p887mm.svg", "w").write("\n".join(out) + "\n")

# the labelled panel: the box, the colorbar centred beside it, the title and n above, the gene names to the left
ET.register_namespace("", NS)
size = lambda root: tuple(float(root.get(k).removesuffix("mm")) for k in ("width", "height"))
box = ET.parse(f"{F}/heatmap_monocle_induced_z3_box_30x59p887mm.svg").getroot()
cb = ET.parse(f"{F}/heatmap_monocle_z3_colorbar_5pt_30x59p887mm.svg").getroot()
(W_BOX, H), (cbw, cbh) = size(box), size(cb)
bar = list(cb.iter(f"{{{NS}}}rect"))
bar_y0 = min(float(r.get("y")) for r in bar)
bar_h = max(float(r.get("y")) + float(r.get("height")) for r in bar) - bar_y0
cb_x, cb_y = W_BOX + 2.0, (H - bar_h) / 2 - bar_y0
labels, pitch, cap = [], H / len(genes), ext("H", 5).height
left = min(0.0, -(0.8 + max(ext(g, 5, italic=True).x1 for g in genes) + 0.3))
for i, g in enumerate(genes):
    labels.append(("genes", {"x": fmt(-0.8), "y": fmt((i + 0.5) * pitch + cap / 2), "text-anchor": "end",
                             "font-style": "italic", "font-size": fmt(5 * PT)}, g))
y = -0.8                                        # baselines stack upward from just above the box
for grp, text, pt in (("subtitles", f"n = {n_induced}", 5), ("titles", "Induced timepoints", 6)):
    y -= -ext(text, pt).y0                      # room for descenders below the baseline
    labels.append((grp, {"x": fmt(W_BOX / 2), "y": fmt(y), "text-anchor": "middle", "font-size": fmt(pt * PT)}, text))
    y -= ext(text, pt).y1 + 0.4                 # ascent + line gap
top, right, bottom = min(0.0, cb_y, y - 0.2), cb_x + cbw, max(H, cb_y + cbh)
svg = ET.Element(f"{{{NS}}}svg", {"width": f"{fmt(right - left)}mm", "height": f"{fmt(bottom - top)}mm",
                                  "viewBox": f"{fmt(left)} {fmt(top)} {fmt(right - left)} {fmt(bottom - top)}"})
svg.append(ET.Comment(" panels left to right: induced, colorbar; gap 2 mm; units mm; boxes start at x = 0, y = 0 "))
for gid, root, gx, gy in (("induced", box, 0.0, 0.0), ("colorbar", cb, cb_x, cb_y)):
    g = ET.SubElement(svg, f"{{{NS}}}g", {"id": gid, "transform": f"translate({fmt(gx)},{fmt(gy)})"})
    for child in root:
        child = copy.deepcopy(child)
        for el in child.iter():                 # keep ids unique across panels
            if el.get("id"):
                el.set("id", f"{gid}_{el.get('id')}")
        g.append(child)
for grp in dict.fromkeys(grp for grp, _, _ in labels):
    g = ET.SubElement(svg, f"{{{NS}}}g", {"id": grp, "font-family": FF, "fill": "#000000"})
    for _, attrs, text in (l for l in labels if l[0] == grp):
        ET.SubElement(g, f"{{{NS}}}text", attrs).text = text
ET.ElementTree(svg).write(f"{F}/heatmap_monocle_induced_labelled_30x59p887mm.svg", encoding="UTF-8",
                          xml_declaration=True)
SVG(filename=f"{F}/heatmap_monocle_induced_labelled_30x59p887mm.svg")

In [ ]:
# Fig. 5F: the 28 conserved genes, human (left) and mouse (right), rows in drawing order
mpl.rc_file_defaults()                          # matplotlib's settings as in a new Python session
plt.rcParams.update({"svg.hashsalt": "xmens-pseudotime-heatmaps",
                     "font.family": ["Helvetica Neue", "Helvetica", "Nimbus Sans", "DejaVu Sans"],
                     "svg.fonttype": "none", "pdf.fonttype": 42, "axes.linewidth": 0.5,
                     "xtick.major.width": 0.5, "xtick.major.size": 2.0, "xtick.major.pad": 1.5})
genes = pd.read_csv(f"{OUT}/inputs/fig5f_genes.csv")
MM, W_MM, H_MM = 1 / 25.4, 142.5, 150.4
fig = plt.figure(figsize=(W_MM * MM, H_MM * MM))
axes_mm = lambda x, y_top, w, h: fig.add_axes([x / W_MM, 1 - (y_top + h) / H_MM, w / W_MM, h / H_MM])
norm, cmap, ng = mpl.colors.Normalize(-3.0, 3.0), plt.get_cmap("RdBu_r"), len(genes)
for species, x0 in (("human", 18.17), ("mouse", 54.17)):
    names = genes[species].astype(str).tolist()
    z = pd.read_csv(f"{OUT}/z/z_{species}.tsv", sep="\t", index_col=0).drop(columns="fit").loc[names]
    ax = axes_mm(x0, 14.59, 32.0, 120.0)
    ax.pcolormesh(np.linspace(0, 1, 101), np.arange(ng + 1), np.clip(z.to_numpy(float), -3.0, 3.0), cmap=cmap,
                  norm=norm, shading="flat", antialiased=False, linewidth=0, edgecolors="none")
    ax.set_xlim(0, 1); ax.set_ylim(ng, 0)
    ax.set_xticks([0, 0.5, 1]); ax.set_xticklabels(["0", "0.5", "1"], fontsize=6)
    ax.set_yticks(np.arange(ng) + 0.5); ax.set_yticklabels(names, fontsize=6, style="italic")
    ax.tick_params(axis="y", length=0, pad=3.5)
    if species == "mouse":
        ax.yaxis.tick_right()
    fig.text(x0 / W_MM, 1 - (14.59 - 1.5) / H_MM, species.capitalize(), fontsize=8, ha="left", va="baseline")
fig.text((18.17 + 54.17 + 32.0) / 2 / W_MM, 1 - 146.0 / H_MM, "pseudotime (rank-normalized)", fontsize=8,
         ha="center", va="baseline")
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=axes_mm(102.17, 105.79, 3.0, 26.4),
                  extend="neither")
cb.set_ticks(np.arange(-3, 4)); cb.set_ticklabels([("−" if t < 0 else "") + str(abs(t)) for t in range(-3, 4)])
cb.ax.tick_params(labelsize=6, width=0.5, length=2, pad=1.5); cb.outline.set_linewidth(0.5)
cb.set_label("z per gene, clipped at ±3\n(Monocle: log10, df = 3 spline)", fontsize=8)
out = f"{OUT}/figures/fig5f/crossspecies28_monocle_clip3_final_objects_nostrips"
for ext, meta in (("svg", {"Date": None}), ("pdf", {"CreationDate": None})):
    fig.savefig(f"{out}.{ext}", metadata=meta, bbox_inches="tight", pad_inches=0.02)
fig.savefig(f"{out}.png", dpi=600, bbox_inches="tight", pad_inches=0.02)

In [ ]:
# Big MI heatmaps: every top MI gene of each species, rows in order of pseudotime centroid (centre of mass of the
# positive part of z; ties by the point of the maximum)
mpl.rc_file_defaults()                          # matplotlib's settings as in a new Python session
plt.rcParams.update({"svg.hashsalt": "xmens-pseudotime-heatmaps",
                     "font.family": ["Helvetica Neue", "Helvetica", "DejaVu Sans"],
                     "svg.fonttype": "none", "pdf.fonttype": 42, "axes.linewidth": 0.5,
                     "xtick.major.width": 0.5, "xtick.major.size": 2.0, "xtick.major.pad": 1.5})
MM, W_MM, H_MM = 1 / 25.4, 160.0, 156.0
fig = plt.figure(figsize=(W_MM * MM, H_MM * MM))
axes_mm = lambda x, y_top, w, h: fig.add_axes([x / W_MM, 1 - (y_top + h) / H_MM, w / W_MM, h / H_MM])
norm, cmap = mpl.colors.Normalize(-3.0, 3.0), plt.get_cmap("RdBu_r")
out = f"{OUT}/figures/mi/mi_top_genes_heatmap_centroid_order"
for species, x0 in (("human", 16.0), ("mouse", 80.0)):
    genes = pd.read_csv(f"{OUT}/inputs/mi_top_{species}.csv")
    z = pd.read_csv(f"{OUT}/z/z_{species}.tsv", sep="\t", index_col=0).drop(columns="fit").loc[genes.gene]
    z = z.to_numpy(float)
    positive = np.clip(z, 0, None)
    centroid = (positive * np.linspace(0, 1, z.shape[1])).sum(1) / positive.sum(1)
    order = np.lexsort((z.argmax(1), centroid))
    n_cells = len(pd.read_csv(f"{OUT}/inputs/cells_{species}.tsv.gz", sep="\t", usecols=["cell"]))
    heat = axes_mm(x0, 8.0, 52.0, 136.0)
    heat.imshow(np.clip(z[order], -3.0, 3.0), cmap=cmap, norm=norm, aspect="auto", interpolation="nearest",
                extent=(0, 1, len(genes), 0), rasterized=True)
    heat.set_xlim(0, 1); heat.set_ylim(len(genes), 0)
    heat.set_xticks([0, 0.5, 1]); heat.set_xticklabels(["0", "0.5", "1"], fontsize=6)
    heat.set_yticks([])
    heat.set_xlabel("pseudotime (rank-normalized)", fontsize=8, labelpad=2)
    heat.set_title(f"{species.capitalize()} ({n_cells:,} nuclei)", loc="left", fontsize=8, pad=3)
    heat.set_ylabel(f"{len(genes):,} genes, ordered by pseudotime centroid", fontsize=7, labelpad=2)
    table = genes.assign(centroid=centroid).iloc[order].reset_index(drop=True)
    table.assign(row=np.arange(1, len(table) + 1)).to_csv(f"{out}_gene_order_{species}.csv", index=False)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=axes_mm(138.0, 110.0, 3.0, 30.0))
cb.set_ticks(np.arange(-3, 4)); cb.set_ticklabels([("−" if t < 0 else "") + str(abs(t)) for t in range(-3, 4)])
cb.ax.tick_params(labelsize=6, width=0.5, length=2, pad=1.5); cb.outline.set_linewidth(0.5)
cb.set_label("expression, z per gene\n(clipped at ±3)", fontsize=7)
for ext, meta in (("svg", {"Date": None}), ("pdf", {"CreationDate": None})):
    fig.savefig(f"{out}.{ext}", dpi=600, metadata=meta)
fig.savefig(f"{out}.png", dpi=600)